# Tutorial 01: Propagación de una Partícula Libre de Dirac (1D, 2D y 3D)

En este tutorial estudiaremos la evolución temporal de un fermión relativista libre ($V(\mathbf{r}) = 0$) gobernado por la **ecuación de Dirac** en unidades naturales ($\hbar = c = m_e = 1$):

$$i \frac{\partial \psi(\mathbf{r}, t)}{\partial t} = \left( -i \boldsymbol{\alpha} \cdot \nabla + \beta m \right) \psi(\mathbf{r}, t)$$

donde $\psi(\mathbf{r}, t)$ es un biespinor de Dirac de 4 componentes complejas y $\boldsymbol{\alpha} = (\alpha_x, \alpha_y, \alpha_z)$, $\beta$ son las matrices de Dirac de dimensión $4 \times 4$.

### Objetivos de aprendizaje
1. Configurar mallas espaciales (`Grid`) en **1D, 2D y 3D** y verificar la condición de estabilidad numérica de Courant-Friedrichs-Lewy (`grid.courant_limit()`).
2. Construir estados iniciales espinoriales (`ConstantSpinor`) modulados como paquetes de onda gaussianos (`GaussianPacket`) y ondas planas (`PlaneWave`).
3. Comparar **condiciones de frontera periódicas** (`PeriodicBoundary`) frente a **condiciones de frontera absorbentes** (`AbsorbingBoundary`).
4. Analizar la dispersión relativista del paquete, la velocidad de grupo $v_g = p / E$ y las oscilaciones de alta frecuencia (*Zitterbewegung*) mediante `Observables`.

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt

# Asegurar que dirac_solver sea localizable desde examples/, examples/es/ o examples/en/
for _rel in ['..', '../..']:
    _abs = os.path.abspath(_rel)
    if os.path.exists(os.path.join(_abs, 'dirac_solver')) and _abs not in sys.path:
        sys.path.insert(0, _abs)

from dirac_solver import (
    Grid,
    ConstantSpinor,
    GaussianPacket,
    PlaneWave,
    DiracProblemBuilder,
    DiracSolver,
    FreeParticle,
    ScalarPotential,
    CoulombPotential,
    YukawaPotential,
    InfiniteWellPotential,
    Observables,
    HDF5Storage,
    _core,
    electron_mass,
    c,
    hbar,
)
from dirac_solver.storage import create_animation, create_animation_1d

plt.rcParams.update({'figure.figsize': (10, 5), 'font.size': 12})

---
## 1. Geometría Espacial y Condición de Estabilidad de Courant (CFL)

El motor C++ `FDTDLeapfrogIntegrator` integra la ecuación de Dirac mediante diferencias finitas en el dominio del tiempo (esquema *Leapfrog* escalonado). Para garantizar estabilidad numérica en una malla de espaciado $(\Delta x, \Delta y, \Delta z)$, el paso temporal $\Delta t$ debe satisfacer la condición CFL:

$$\Delta t \le \Delta t_{\text{CFL}} = \frac{1}{c \sqrt{\sum_{k=1}^{D} \frac{1}{\Delta x_k^2}}}$$

El método `grid.courant_limit()` calcula automáticamente este límite.

In [ ]:
# Definición de mallas en 1D, 2D y 3D
grid_1D = Grid(shape=(240,), spacing=(0.1,))
grid_2D = Grid(shape=(80, 80), spacing=(0.15, 0.15))
grid_3D = Grid(shape=(24, 24, 24), spacing=(0.25, 0.25, 0.25))

for name, g in [('1D', grid_1D), ('2D', grid_2D), ('3D', grid_3D)]:
    print(f"Malla {name}: shape={g.shape}, puntos={g.get_total_points()}, "
          f"dV={g.get_cell_volume():.4f}, dt_max (Courant)={g.courant_limit():.5f}")

---
## 2. Estados Iniciales: `ConstantSpinor`, `GaussianPacket` y `PlaneWave`

Un espinor de energía positiva con momento medio $\mathbf{p}_0$ y polarización de espín $\hat{\mathbf{n}}$ tiene la estructura:

$$u(\mathbf{p}_0) = \begin{pmatrix} \sqrt{E + m}\, \chi \\ \sqrt{E - m}\, \frac{\boldsymbol{\sigma} \cdot \mathbf{p}_0}{|\mathbf{p}_0|} \chi \end{pmatrix}, \quad E = \sqrt{|\mathbf{p}_0|^2 + m^2}$$

- **`GaussianPacket`**: $\psi(\mathbf{r}, 0) = u(\mathbf{p}_0) \, e^{i \mathbf{p}_0 \cdot \mathbf{r}} \exp\!\left(-\frac{|\mathbf{r} - \mathbf{r}_0|^2}{4\sigma_0^2}\right)$
- **`PlaneWave`**: $\psi(\mathbf{r}, 0) = u(\mathbf{p}_0) \, e^{i \mathbf{p}_0 \cdot \mathbf{r}}$

In [ ]:
# 1D: momento a lo largo del eje x
spinor_1D = ConstantSpinor(spin_polarization=[0, 0, 1], momentum=[3.0, 0.0, 0.0])
initial_1D_GP = GaussianPacket(spinor_1D, center=[-2.0, 0.0, 0.0], spatial_width=1.0)
initial_1D_PW = PlaneWave(spinor_1D)

# 2D: momento diagonal en el plano xy
spinor_2D = ConstantSpinor(spin_polarization=[0, 0, 1], momentum=[3.0, 3.0, 0.0])
initial_2D_GP = GaussianPacket(spinor_2D, center=[-1.5, -1.5, 0.0], spatial_width=1.0)

# 3D: momento en las tres direcciones espaciales
spinor_3D = ConstantSpinor(spin_polarization=[0, 0, 1], momentum=[2.5, 2.5, 2.5])
initial_3D_GP = GaussianPacket(spinor_3D, center=[0.0, 0.0, 0.0], spatial_width=0.9)

---
## 3. Simulación 1D: Frontera Periódica vs. Frontera Absorbente

Cuando el paquete gaussiano alcanza el borde del dominio computacional:
- Con **`PeriodicBoundary`**, el paquete reingresa por el extremo opuesto (topología de anillo).
- Con **`AbsorbingBoundary(strength, width)`**, una capa amortiguadora atenúa suavemente la función de onda cerca de los bordes para evitar reflexiones espurias.

In [ ]:
potential_free = FreeParticle()
boundary_PB = _core.PeriodicBoundary()
boundary_AB = _core.AbsorbingBoundary(strength=0.4, width=25)

dt_1d = 0.005  # Menor que grid_1D.courant_limit() = 0.1
total_time_1d = 7.0

# Construcción del problema con frontera periódica (PB)
problem_1D_PB = (
    DiracProblemBuilder()
    .set_grid(grid_1D)
    .set_initial_state(initial_1D_GP)
    .set_potential(potential_free)
    .set_boundary_condition(boundary_PB)
    .set_time_parameters(time_step=dt_1d, total_time=total_time_1d)
    .build()
)
solver_1D_PB = DiracSolver(problem_1D_PB)
obs_1D_PB = Observables(solver_1D_PB)

# Construcción del problema con frontera absorbente (AB)
problem_1D_AB = (
    DiracProblemBuilder()
    .set_grid(grid_1D)
    .set_initial_state(initial_1D_GP)
    .set_potential(potential_free)
    .set_boundary_condition(boundary_AB)
    .set_time_parameters(time_step=dt_1d, total_time=total_time_1d)
    .build()
)
solver_1D_AB = DiracSolver(problem_1D_AB)
obs_1D_AB = Observables(solver_1D_AB)

# Guardamos la densidad inicial para comparar
x_1d = grid_1D.coords[:, 0]
rho_init_1d = obs_1D_PB.probability_density(solver_1D_PB.get_psi())

# Ejecutamos ambas simulaciones registrando observables
solver_1D_PB.run_simulation(observables_handler=obs_1D_PB)
solver_1D_AB.run_simulation(observables_handler=obs_1D_AB)

In [ ]:
# Comparación gráfica de densidades de probabilidad y norma total
rho_final_PB = obs_1D_PB.probability_density(solver_1D_PB.get_psi())
rho_final_AB = obs_1D_AB.probability_density(solver_1D_AB.get_psi())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(x_1d, rho_init_1d, 'k--', alpha=0.6, label='Inicial (t = 0)')
axes[0].plot(x_1d, rho_final_PB, 'b-', lw=2, label=f'Periódica (t = {total_time_1d})')
axes[0].plot(x_1d, rho_final_AB, 'r-', lw=2, label=f'Absorbente (t = {total_time_1d})')
axes[0].axvspan(x_1d[-25], x_1d[-1], color='gray', alpha=0.2, label='Zona absorbente')
axes[0].axvspan(x_1d[0], x_1d[24], color='gray', alpha=0.2)
axes[0].set_xlabel('Posición x')
axes[0].set_ylabel(r'Densidad de probabilidad $\rho(x,t) = \psi^\dagger \psi$')
axes[0].set_title('Propagación 1D: Frontera Periódica vs. Absorbente')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Evolución de la posición esperada <x>(t) y velocidad de grupo teórica
t_hist = obs_1D_PB.get_history('time')
x_exp_PB = obs_1D_PB.get_history('expected_position')[:, 0]
p0 = 3.0
v_group = p0 / np.sqrt(p0**2 + electron_mass**2)

axes[1].plot(t_hist, x_exp_PB, 'b-', lw=2, label=r'$\langle x \rangle(t)$ numérico (PB)')
axes[1].plot(t_hist, -2.0 + v_group * t_hist, 'g--', lw=2, label=f'Teórico $-2 + v_g t$ ($v_g={v_group:.3f}c$)')
axes[1].set_xlabel('Tiempo t')
axes[1].set_ylabel(r'Posición esperada $\langle x \rangle$')
axes[1].set_title('Trayectoria del Centro del Paquete y Reentrada Periódica')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 4. Propagación en 2D y 3D

El método `solver.plot_probability_density()` adapta automáticamente la visualización según la dimensión de la malla:
- **1D**: Curva de línea $\rho(x)$.
- **2D**: Mapa de calor $\rho(x, y)$.
- **3D**: Nube volumétrica de puntos con tamaño y opacidad proporcionales a $\rho(x, y, z)$.

In [ ]:
# Simulación 2D con Paquete Gaussiano
problem_2D = (
    DiracProblemBuilder()
    .set_grid(grid_2D)
    .set_initial_state(initial_2D_GP)
    .set_potential(potential_free)
    .set_boundary_condition("periodic")
    .set_time_parameters(time_step=0.005, total_time=1.5)
    .build()
)
solver_2D = DiracSolver(problem_2D)
solver_2D.run_simulation()
solver_2D.plot_probability_density()

In [ ]:
# Simulación 3D con Paquete Gaussiano
problem_3D = (
    DiracProblemBuilder()
    .set_grid(grid_3D)
    .set_initial_state(initial_3D_GP)
    .set_potential(potential_free)
    .set_boundary_condition("periodic")
    .set_time_parameters(time_step=0.01, total_time=0.6)
    .build()
)
solver_3D = DiracSolver(problem_3D)
solver_3D.run_simulation()
solver_3D.plot_probability_density()